In [11]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import random

In [12]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

def sim_bb84_plain(num_bits):
  # Alice generates random sequence of qubits
    # represents each bit by a qubit using one of two schemes:
    # (a) represent 0 by |0>  and 1 by |1>   (the standard basis)
    # (b) represent 0 by |+>  and 1 by |->    (the diagonal basis)
    # chosen randomly and independently for each qubit.
  alice_bits = [random.randint(0, 1) for _ in range(num_bits)]
  alice_bases = [random.randint(0, 1) for _ in range(num_bits)]

  # Alice prepares and sends resulting qubit sequence to Bob
  qc = QuantumCircuit(num_bits, num_bits)
  for i in range(num_bits):
    # Encode bit value: 1 requires an X gate
    if alice_bits[i] == 1:
      qc.x(i) # To represent bit value of 1 using |1>
    # Encode basis: Diagonal requires a Hadamard (H) gate
    if alice_bases[i] == 1:
      qc.h(i)

  # For each qubit, Bob measure in either std or diagonal basis, chosen randomly and independently for each qubit
  bob_bases = [random.randint(0, 1) for _ in range(num_bits)]
  for i in range(num_bits):
    if bob_bases[i] == 1: # if diagonal, rotate back before measuring
      qc.h(i)
    qc.measure(i, i)

  # Execution
  backend = BasicSimulator()
  # shots=1 as each qubit is transmitted once
  # memory=True keeps record each individual measurement result
  job = backend.run(transpile(qc, backend), shots=1, memory=True)
  bob_results = [int(bit) for bit in job.result().get_memory()[0][::-1]]

  # Bob tells Alice his sequence of basis choices
  # Alice tells Bob which ones match her choices
  shared_key_alice = []
  shared_key_bob = []

  # For any given qubit,
  for i in range(num_bits):
    if alice_bases[i] == bob_bases[i]: # Only keep if bases matched (correct bit)
      shared_key_alice.append(alice_bits[i])
      shared_key_bob.append(bob_results[i])

  print(f"Bases matched for {len(shared_key_alice)} out of {num_bits} bits.")
  print(f"Keys Match: {shared_key_alice == shared_key_bob}")
  return shared_key_alice

key = sim_bb84_plain(20)
print(f"Final Shared Key: {key}")

Bases matched for 12 out of 20 bits.
Keys Match: True
Final Shared Key: [0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1]
